<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/02-deteccao-em-video.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 2. Detecção quadro a quadro em vídeo

Vídeo é só uma sequência de fotos. Para detectar objetos nele, você lê um quadro, roda o modelo, desenha as caixas e grava o quadro num arquivo novo. Repete até o fim.

O exemplo é um clipe de cinco segundos (menos de 1 MB) em que a câmera passeia por uma sala com cadeiras, plantas, mesa e livros. Muita classe ao mesmo tempo, e roda rápido até em CPU.

> GPU deixa bem mais rápido: **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU**.

In [1]:
!pip install -q ultralytics

In [2]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

In [3]:
from collections import Counter

import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
video = baixar(
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/home-objects-compressed.mp4",
    "sala.mp4",
)

sala.mp4: 0.7 MB


## Ler, detectar, gravar

O OpenCV lê o vídeo com `VideoCapture` e escreve com `VideoWriter`. A resolução e o FPS do arquivo de saída precisam bater com os do original, senão o vídeo sai corrompido.

In [4]:
MAX_QUADROS = 150

cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{largura}x{altura} a {fps} FPS")

out = cv2.VideoWriter("sala_detectada.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura))
contagem = Counter()
quadros = 0

while quadros < MAX_QUADROS:
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, verbose=False)[0]
    contagem.update(model.names[int(c)] for c in r.boxes.cls)
    out.write(r.plot())
    quadros += 1

cap.release()
out.release()
print(f"{quadros} quadros processados")
assert quadros > 0

1280x720 a 30 FPS


140 quadros processados


## Assistir o resultado

In [5]:
web = mostrar_video("sala_detectada.mp4")

## Quantas detecções de cada classe?

Repare que o número é de **detecções**, não de objetos. A mesma cadeira aparece em dezenas de quadros e é contada em cada um. Separar "a mesma cadeira" de "outra cadeira" é exatamente o que o rastreamento resolve na próxima aula.

In [6]:
for classe, n in contagem.most_common():
    print(f"{classe:12s} {n:5d} detecções  (~{n / quadros:.1f} por quadro)")

chair          311 detecções  (~2.2 por quadro)
potted plant   259 detecções  (~1.9 por quadro)
dining table   121 detecções  (~0.9 por quadro)
vase            79 detecções  (~0.6 por quadro)
book            65 detecções  (~0.5 por quadro)
cup             34 detecções  (~0.2 por quadro)
laptop          27 detecções  (~0.2 por quadro)
bottle          15 detecções  (~0.1 por quadro)
tv               3 detecções  (~0.0 por quadro)


## Baixar o vídeo

No Colab, a célula abaixo baixa o vídeo anotado para o seu computador.

In [7]:
baixar_para_maquina(web)

Fora do Colab: o arquivo está em sala_detectada_web.mp4


## Exercícios

Seis exercícios para fixar a aula. Escreva a resposta na variável `resposta` e rode a célula: a função `conferir` diz na hora se está certo. Se travar, abra **Ver solução**.

Rode antes a célula de preparação abaixo. Ela só carrega as verificações.

In [8]:
# @title Preparação dos exercícios (rode, não precisa ler)
from functools import lru_cache

_cap = cv2.VideoCapture(video)
_total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
_fps = _cap.get(cv2.CAP_PROP_FPS)
_cap.release()


@lru_cache(None)
def _classes_por_quadro(n):
    cap = cv2.VideoCapture(video)
    saida = []
    for _ in range(n):
        ok, frame = cap.read()
        if not ok:
            break
        saida.append(tuple(model.names[int(c)] for c in model(frame, verbose=False)[0].boxes.cls))
    cap.release()
    return saida


def _quadros_do_arquivo(caminho):
    cap = cv2.VideoCapture(str(caminho))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


VERIFICAR = {
    1: lambda r: abs(float(r) - _total / _fps) < 0.05,
    2: lambda r: r == contagem.most_common(1)[0][0],
    3: lambda r: int(r) == sum(q.count("chair") for q in _classes_por_quadro(60)[:50:5]),
    4: lambda r: int(r) == sum("chair" in q for q in _classes_por_quadro(60)),
    5: lambda r: _quadros_do_arquivo(r) == 60,
    6: lambda r: abs(float(r) - contagem["chair"] / quadros) < 1e-3,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

### Exercício 1. Duração do vídeo

Quantos segundos tem o vídeo **inteiro**? Use o total de quadros e o FPS do arquivo.

<details>
<summary>Ver solução</summary>

```python
cap = cv2.VideoCapture(video)
resposta = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
cap.release()
```

</details>

In [9]:
# Dica: cap.get(cv2.CAP_PROP_FRAME_COUNT) e cap.get(cv2.CAP_PROP_FPS).
resposta = None

conferir(1, resposta)

Exercício 1: ainda sem resposta.


### Exercício 2. A classe campeã

Qual classe teve mais detecções no vídeo processado? Responda com o nome.

<details>
<summary>Ver solução</summary>

```python
resposta = contagem.most_common(1)[0][0]
```

</details>

In [10]:
# Dica: contagem é um Counter, e Counter tem most_common().
resposta = None

conferir(2, resposta)

Exercício 2: ainda sem resposta.


### Exercício 3. Pular quadros

Processar todo quadro é caro. Nos primeiros 50 quadros, rode o modelo só em 1 a cada 5 (quadros 0, 5, 10...) e some quantas cadeiras (`chair`) foram detectadas.

<details>
<summary>Ver solução</summary>

```python
cap = cv2.VideoCapture(video)
resposta = 0
for i in range(50):
    ok, frame = cap.read()
    if not ok:
        break
    if i % 5 == 0:
        r = model(frame, verbose=False)[0]
        resposta += sum(model.names[int(c)] == "chair" for c in r.boxes.cls)
cap.release()
```

</details>

In [11]:
# Dica: Leia todos os quadros com cap.read(), mas só chame o modelo quando i % 5 == 0.
resposta = None

conferir(3, resposta)

Exercício 3: ainda sem resposta.


### Exercício 4. Quadros com cadeira

Em quantos dos primeiros 60 quadros aparece **pelo menos uma** cadeira?

<details>
<summary>Ver solução</summary>

```python
cap = cv2.VideoCapture(video)
resposta = 0
for _ in range(60):
    ok, frame = cap.read()
    if not ok:
        break
    nomes = [model.names[int(c)] for c in model(frame, verbose=False)[0].boxes.cls]
    resposta += "chair" in nomes
cap.release()
```

</details>

In [12]:
# Dica: Para cada quadro, monte a lista de nomes e teste "chair" in nomes.
resposta = None

conferir(4, resposta)

Exercício 4: ainda sem resposta.


### Exercício 5. Vídeo só com plantas

Grave um vídeo novo com os primeiros 60 quadros, desenhando só as detecções de `potted plant`. A resposta é o caminho do arquivo. Descubra o índice da classe em `model.names`.

<details>
<summary>Ver solução</summary>

```python
planta = [k for k, v in model.names.items() if v == "potted plant"][0]
cap = cv2.VideoCapture(video)
out = cv2.VideoWriter("so_plantas.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (largura, altura))
for _ in range(60):
    ok, frame = cap.read()
    if not ok:
        break
    out.write(model(frame, classes=[planta], verbose=False)[0].plot())
cap.release()
out.release()
resposta = "so_plantas.mp4"
mostrar_video(resposta)
```

</details>

In [13]:
# Dica: Passe classes=[índice] na chamada do modelo e use o mesmo VideoWriter da aula.
resposta = None

conferir(5, resposta)

Exercício 5: ainda sem resposta.


### Exercício 6. Média por quadro

Em média, quantas cadeiras o modelo detectou por quadro no vídeo processado?

<details>
<summary>Ver solução</summary>

```python
resposta = contagem["chair"] / quadros
```

</details>

In [14]:
# Dica: Você já tem contagem e quadros.
resposta = None

conferir(6, resposta)

Exercício 6: ainda sem resposta.


In [15]:
placar()

0 de 6 exercícios certos.


## O que fica

Detecção em vídeo é um laço: ler, inferir, desenhar, gravar. O gargalo é a inferência, por isso GPU e modelos menores fazem tanta diferença. Para filtrar só uma classe, passe `classes=[0]` (pessoa) na chamada do modelo.